# Video Summarisation + Quiz Generation (NLP Project)

## 🎯 Goal
End-to-end pipeline for **educational lecture videos**:

**Video → Audio → Transcript → Clean + Sentence split → Deduplicate → Summarise → Quiz → Evaluate → Final demo display**

## ✅ Important notes (for demo)
- Works best on a **short video (3–7 min)**.
- Use **GPU runtime**: Runtime → Change runtime type → **T4 GPU**.
- This notebook uses **pretrained models** (no fine-tuning required for a working demo). Fine-tuning is future work.


## 0) Folder structure (auto-created)
- `youtube_videos/` downloaded YouTube `.mp4`
- `audio_files/` extracted `.wav` (16kHz mono)
- `transcripts/` raw transcripts `.txt`
- `segmented/` sentence-tokenized + cleaned `.txt`
- `deduplicated/` redundancy-removed sentences `.txt`
- `summaries/` extractive summaries `.txt`
- `quiz_output/` quizzes `.json`
- `evaluation/` evaluation report `.json`


In [ ]:
# CELL 1: Install all dependencies (run once per runtime)

!pip install -q kagglehub yt-dlp faster-whisper sentence-transformers nltk rouge-score transformers sentencepiece gensim

import os
import re
import json
import time
import random
import subprocess
import torch
import nltk

from nltk.tokenize import sent_tokenize
from nltk.corpus import wordnet, stopwords
from rouge_score import rouge_scorer
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

random.seed(42)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

print('✅ Installs + imports complete')
print('GPU:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# CELL 2: Choose input source (YouTube recommended for demo)
# Set SOURCE = 'youtube' OR 'kaggle'

SOURCE = 'youtube'  # 'youtube' or 'kaggle'

print('SOURCE set to:', SOURCE)

In [ ]:
# CELL 3A (YouTube): Download 1 lecture video
# Run this cell only if SOURCE == 'youtube'

import yt_dlp

if SOURCE != 'youtube':
    print('Skipping YouTube download (SOURCE != youtube).')
else:
    # ─── CHANGE THIS URL BEFORE DEMO ───────────────────────────────
    YOUTUBE_URL = 'https://www.youtube.com/watch?v=aircAruvnKk'  # 3Blue1Brown: Neural Networks
    # ──────────────────────────────────────────────────────────────

    youtube_dir = 'youtube_videos'
    os.makedirs(youtube_dir, exist_ok=True)

    print('Downloading video from:', YOUTUBE_URL)

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': f'{youtube_dir}/%(title).50s.%(ext)s',
        'quiet': False,
        'merge_output_format': 'mp4',
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(YOUTUBE_URL, download=True)
        print('\n✅ Downloaded:', info.get('title'))

    video_files = [
        os.path.join(youtube_dir, f)
        for f in os.listdir(youtube_dir)
        if f.endswith('.mp4')
    ]

    print('Video ready:', video_files)

In [ ]:
# CELL 3B (Kaggle): Download dataset and pick first N videos
# Run this cell only if SOURCE == 'kaggle'

if SOURCE != 'kaggle':
    print('Skipping Kaggle dataset (SOURCE != kaggle).')
else:
    import kagglehub

    path = kagglehub.dataset_download('pad19tue/lecture-videos')
    video_dir = os.path.join(path, 'video')

    all_videos = sorted([
        os.path.join(video_dir, v)
        for v in os.listdir(video_dir)
        if v.endswith('.mp4')
    ])

    N = 3
    video_files = all_videos[:N]

    print('✅ Kaggle path:', path)
    print(f'Using {len(video_files)} videos:', video_files[:2], '...')

In [ ]:
# CELL 4: Extract audio (16kHz mono wav)

audio_dir = 'audio_files'
os.makedirs(audio_dir, exist_ok=True)

audio_files = []

for video_path in video_files:
    video_name = os.path.basename(video_path)
    audio_path = os.path.join(audio_dir, video_name.replace('.mp4', '.wav'))

    if os.path.exists(audio_path):
        print('Audio exists, skipping:', audio_path)
        audio_files.append(audio_path)
        continue

    print('Extracting audio from:', video_name)
    cmd = ['ffmpeg', '-y', '-i', video_path, '-ac', '1', '-ar', '16000', audio_path]
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)

    if result.returncode == 0:
        audio_files.append(audio_path)
        print('✅ Saved:', audio_path)
    else:
        print('❌ ffmpeg error for', video_name)
        print(result.stderr.decode(errors='ignore')[:4000])

print('\nTotal audio files:', len(audio_files))

In [ ]:
# CELL 5: Transcribe audio with faster-whisper (base)
# Notes:
# - 'base' improves accuracy vs 'tiny' (slower, but better technical vocab).
# - beam_size=1 is fastest.

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'

model_faster = WhisperModel('base', device=device, compute_type=compute_type)
print(f'✅ WhisperModel loaded on {device} ({compute_type})')

transcript_dir = 'transcripts'
os.makedirs(transcript_dir, exist_ok=True)

for idx, audio_path in enumerate(audio_files, start=1):
    out_path = os.path.join(transcript_dir, os.path.basename(audio_path).replace('.wav', '.txt'))

    if os.path.exists(out_path):
        print(f'[{idx}] Transcript exists, skipping:', out_path)
        continue

    print(f'[{idx}/{len(audio_files)}] Transcribing:', os.path.basename(audio_path))
    t0 = time.time()
    segments, info = model_faster.transcribe(audio_path, beam_size=1)
    text = ' '.join([seg.text for seg in segments]).strip()

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(text)

    print('✅ Saved:', out_path, '| time:', round(time.time() - t0, 2), 's')
    print('Preview:', text[:180], '...')

print('\n✅ Transcription complete')

In [ ]:
# CELL 6: Clean + sentence split
# Removes common filler words and normalizes whitespace.

segmented_dir = 'segmented'
os.makedirs(segmented_dir, exist_ok=True)

def clean_sentence(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(uh|um|you know|like|basically|actually|right|okay|so well)\b", '', s)
    s = re.sub(r"\s+", ' ', s)
    return s.strip()

for fname in os.listdir(transcript_dir):
    if not fname.endswith('.txt'):
        continue

    in_path = os.path.join(transcript_dir, fname)
    out_path = os.path.join(segmented_dir, fname)

    with open(in_path, 'r', encoding='utf-8') as f:
        text = f.read()

    sents = sent_tokenize(text)
    sents = [clean_sentence(s) for s in sents if len(s.strip()) > 10]

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(sents))

    print(f'✅ {fname}: {len(sents)} cleaned sentences')

print('\n✅ Segmentation complete')

In [ ]:
# CELL 7: Deduplicate sentences (SentenceTransformer cosine similarity)
# Removes repeated / near-duplicate sentences.

dedup_dir = 'deduplicated'
os.makedirs(dedup_dir, exist_ok=True)

model_embed = SentenceTransformer('all-MiniLM-L6-v2')

def remove_redundant(sentences, threshold=0.8):
    if not sentences:
        return []
    emb = model_embed.encode(sentences, convert_to_tensor=True)
    keep_ids = []
    for i in range(len(sentences)):
        redundant = any(util.cos_sim(emb[i], emb[j]).item() > threshold for j in keep_ids)
        if not redundant:
            keep_ids.append(i)
    return [sentences[i] for i in keep_ids]

for fname in os.listdir(segmented_dir):
    if not fname.endswith('.txt'):
        continue
    in_path = os.path.join(segmented_dir, fname)
    out_path = os.path.join(dedup_dir, fname)

    with open(in_path, 'r', encoding='utf-8') as f:
        sents = [ln.strip() for ln in f if ln.strip()]

    filtered = remove_redundant(sents, threshold=0.8)

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(filtered))

    print(f'✅ {fname}: {len(sents)} → {len(filtered)} (removed {len(sents)-len(filtered)})')

print('\n✅ Deduplication complete')

In [ ]:
# CELL 8: Two-stage summarisation — Chunked BART
#
# Pipeline:
#   Deduplicated transcript sentences
#       → split into overlapping chunks of ~20 sentences
#       → BART summarises each chunk independently
#       → chunk summaries joined into one full summary
#       → semantically incoherent sentences filtered using
#         embedding similarity (no hardcoded word lists)
#
# This ensures every part of the video is covered and
# the output reads as proper written prose, not raw transcript.

from transformers import BartForConditionalGeneration, BartTokenizer
from sentence_transformers import SentenceTransformer, util as st_util
import torch, os, re

summary_dir = 'summaries'
os.makedirs(summary_dir, exist_ok=True)

# ── Load BART ─────────────────────────────────────────────────────────
print("Loading BART (facebook/bart-large-cnn)...")
bart_tok = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_mod = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
bart_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bart_mod = bart_mod.to(bart_dev)
bart_mod.eval()
print(f"✅ BART loaded on {bart_dev}")


def prepare_text(sentences):
    """
    Join sentences into a clean paragraph for BART input.
    Fixes basic transcript artifacts universally:
      - Restore sentence capitalisation
      - Fix lone 'i' → 'I'
      - Collapse extra whitespace
    No topic-specific or hardcoded word removal.
    """
    cleaned = []
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        s = s[0].upper() + s[1:]          # capitalise first letter
        s = re.sub(r'\bi\b', 'I', s)      # fix pronoun
        s = re.sub(r'\s+', ' ', s)        # collapse spaces
        cleaned.append(s)
    return ' '.join(cleaned)


def chunk_sentences(sentences, chunk_size=20, overlap=2):
    """
    Split into overlapping chunks so no content is lost at boundaries.
    overlap=2 means last 2 sentences of chunk N become first 2 of chunk N+1.
    """
    chunks, i = [], 0
    while i < len(sentences):
        chunks.append(sentences[i: i + chunk_size])
        i += chunk_size - overlap
    return chunks


def summarise_chunk(sentences, min_len=40, max_len=110):
    """Run BART on one chunk of sentences."""
    text   = prepare_text(sentences)
    inputs = bart_tok(
        text,
        return_tensors='pt',
        max_length=1024,
        truncation=True
    ).to(bart_dev)

    with torch.no_grad():
        ids = bart_mod.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_len,
            min_length=min_len,
            num_beams=4,
            length_penalty=1.5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    return bart_tok.decode(ids[0], skip_special_tokens=True).strip()


def semantic_filter(sentences, embed_model, min_sim=0.10):
    """
    Remove sentences that are semantically incoherent with the rest.

    Strategy: compute each sentence's average cosine similarity to all
    other sentences. Sentences with avg_sim below (mean - 1 std dev)
    are outliers — they don't belong to the main topic.

    min_sim=0.10 is intentionally low so only genuine outliers are removed.
    No word lists, no hardcoded patterns — works for any topic.
    """
    if len(sentences) <= 4:
        return sentences   # too few to filter meaningfully

    embs   = embed_model.encode(sentences, convert_to_tensor=True)
    scores = st_util.cos_sim(embs, embs).mean(dim=1)   # avg sim to all others

    threshold = max(min_sim, scores.mean().item() - scores.std().item())
    kept = [s for s, score in zip(sentences, scores.tolist()) if score >= threshold]

    # Safety: never remove more than 30% of sentences
    if len(kept) < len(sentences) * 0.7:
        kept = sentences
    return kept


def postprocess_filter(sentences):
    """
    Universal post-processing filter using LDA topic coherence.

    How it works:
    1. Train a small LDA model on the summary sentences themselves
       to discover the dominant topic(s) of the content
    2. Score each sentence by how strongly it belongs to the
       dominant topic
    3. Remove sentences whose topic score is a statistical outlier
       (more than 1.5 std devs below the mean)

    Why this is universal:
    - LDA learns topics FROM the content — no hardcoded words
    - Works for neural networks, history, biology, cooking, anything
    - A sponsor mention in ANY video will score low because its
      vocabulary (names, money, companies) doesn't match the
      dominant educational topic of the summary
    - The threshold is statistical (mean - 1.5*std), not fixed

    Safety:
    - Requires at least 6 sentences to run (too few = unreliable)
    - Never removes more than 30% of sentences
    - Falls back to original list if LDA fails for any reason
    """
    if len(sentences) < 6:
        return sentences  # not enough data for reliable topic modelling

    try:
        from gensim import corpora
        from gensim.models import LdaModel
        from nltk.tokenize import word_tokenize
        from nltk.corpus import stopwords as sw
        import string

        stop = set(sw.words('english'))

        def tokenise(sent):
            tokens = word_tokenize(sent.lower())
            return [
                t for t in tokens
                if t not in stop
                and t not in string.punctuation
                and len(t) > 2
                and t.isalpha()
            ]

        tokenised = [tokenise(s) for s in sentences]

        valid_idx = [i for i, t in enumerate(tokenised) if len(t) >= 2]
        if len(valid_idx) < 4:
            return sentences

        valid_tokens = [tokenised[i] for i in valid_idx]
        valid_sents  = [sentences[i] for i in valid_idx]

        dictionary = corpora.Dictionary(valid_tokens)
        corpus     = [dictionary.doc2bow(t) for t in valid_tokens]

        lda = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=2,
            passes=10,
            alpha='auto',
            random_state=42
        )

        scores = []
        for bow in corpus:
            topic_dist = lda.get_document_topics(bow, minimum_probability=0.0)
            dominant_score = max(prob for _, prob in topic_dist)
            scores.append(dominant_score)

        mean_score = sum(scores) / len(scores)
        variance   = sum((s - mean_score) ** 2 for s in scores) / len(scores)
        std_score  = variance ** 0.5
        threshold  = mean_score - 1.5 * std_score

        kept = [
            sent for sent, score in zip(valid_sents, scores)
            if score >= threshold
        ]

        no_content = [sentences[i] for i in range(len(sentences))
                      if i not in valid_idx]
        kept = kept + no_content

        if len(kept) < len(sentences) * 0.70:
            print("  ⚠ Post-process filter too aggressive, keeping original")
            return sentences

        removed = len(sentences) - len(kept)
        if removed > 0:
            print(f"  LDA filter removed {removed} off-topic sentence(s)")

        return kept

    except Exception as e:
        print(f"  ⚠ Post-process filter skipped: {e}")
        return sentences


def prefilter_sentences(sentences, embed_model):
    """
    Remove meta-commentary sentences BEFORE they reach BART.

    These are sentences about the video itself rather than the
    educational content — e.g. "In this video we look at...",
    "The next video will cover...", "Thanks to X for funding..."

    Two universal signals, no hardcoded topic words:

    SIGNAL 1 — Self-referential verb phrases (topic-neutral):
        Sentences where the grammatical subject refers to the video,
        series, or speaker AND the verb is presentational rather than
        explanatory. Detected via POS patterns, not word lists.
        Pattern: sentence starts with a pronoun/demonstrative +
        copula/presentational verb + reference to media/content.
        Universal markers: "this video", "this series", "next video",
        "in this [noun]", "the following [noun]" — these are structural
        phrases about media, not about any subject matter.

    SIGNAL 2 — Proper noun outliers (topic-neutral):
        Sentences containing proper nouns (NNP) that appear in only
        that one sentence AND are not in the majority vocabulary of
        the corpus. These are sponsor/person mentions.
        Detected by: unique NNP ratio > 8% of sentence word count
        AND sentence length > mean length of all sentences.

    Safety: never removes more than 25% of sentences.
    Falls back to original if NLTK tagging fails.
    """
    import re
    import string
    import nltk
    from collections import Counter

    if len(sentences) < 4:
        return sentences

    # ── Signal 1: Self-referential media phrases ──────────────────────
    media_ref_patterns = [
        r'\bthis video\b',
        r'\bthis series\b',
        r'\bnext video\b',
        r'\bfollowing video\b',
        r'\bin this (?:chapter|episode|part|lecture|tutorial)\b',
        r'\bthe following (?:one|video|chapter|part)\b',
        r"\bwe\'re going to\b",
        r'\bwe will\b',
        r"\bwe\'ll\b",
        r"\blet\'s\b",
    ]

    def has_media_ref(sent):
        s = sent.lower()
        for pat in media_ref_patterns:
            if re.search(pat, s):
                return True
        return False

    # ── Signal 2: Unique proper noun density ─────────────────────────
    all_proper = []
    sent_proper = []
    for sent in sentences:
        try:
            tokens = nltk.word_tokenize(sent)
            tags   = nltk.pos_tag(tokens)
            proper = [w.lower() for w, t in tags
                      if t in ('NNP', 'NNPS') and len(w) > 2]
        except Exception:
            proper = []
        sent_proper.append(set(proper))
        all_proper.extend(proper)

    noun_freq   = Counter(all_proper)
    unique_nouns = {n for n, c in noun_freq.items() if c == 1}

    lengths    = [len(s.split()) for s in sentences]
    mean_len   = sum(lengths) / len(lengths)

    def has_sponsor_signal(sent, idx):
        wc           = len(sent.split())
        unique_here  = sent_proper[idx] & unique_nouns
        unique_ratio = len(unique_here) / max(wc, 1)
        return wc > mean_len and unique_ratio > 0.08

    kept = []
    for i, sent in enumerate(sentences):
        if has_media_ref(sent):
            print(f"  Pre-filter (media ref): {sent[:60]}...")
            continue
        if has_sponsor_signal(sent, i):
            print(f"  Pre-filter (sponsor):   {sent[:60]}...")
            continue
        kept.append(sent)

    if len(kept) < len(sentences) * 0.75:
        print("  ⚠ Pre-filter too aggressive, keeping original")
        return sentences

    print(f"  Pre-filter: {len(sentences)} → {len(kept)} sentences")
    return kept


def summarise_full_transcript(sentences, embed_model):
    """
    Full chunked BART pipeline:
    1. Filter very short sentences
    2. Chunk (size=20, overlap=2)
    3. BART summarise each chunk
    4. Join chunk summaries
    5. Optional final compression if >500 words
    6. Semantic filter to remove incoherent artifact sentences
    """
    sents = [s for s in sentences if len(s.split()) >= 5]
    if not sents:
        return ""

    # Pre-filter: remove meta-commentary BEFORE BART sees it
    # This prevents BART from including video structure sentences
    # and sponsor mentions in the summary output
    sents = prefilter_sentences(sents, embed_model)
    if not sents:
        return ""

    print(f"  Input after pre-filter: {len(sents)} sentences")
    chunks = chunk_sentences(sents, chunk_size=20, overlap=2)
    print(f"  Chunks: {len(chunks)}")

    chunk_summaries = []
    for idx, chunk in enumerate(chunks):
        print(f"  Chunk {idx+1}/{len(chunks)}...")
        try:
            s = summarise_chunk(chunk, min_len=40, max_len=110)
            if s:
                chunk_summaries.append(s)
        except Exception as e:
            print(f"  ⚠ Chunk {idx+1} error: {e}")

    if not chunk_summaries:
        return prepare_text(sents[:10])

    full = ' '.join(chunk_summaries)

    # Final compression if too long
    if len(full.split()) > 500:
        print("  Compression pass...")
        try:
            full = summarise_chunk(full.split('. '), min_len=120, max_len=320)
        except Exception:
            pass

    # Semantic filter — removes incoherent sentences, no hardcoded patterns
    from nltk.tokenize import sent_tokenize
    raw_sents   = sent_tokenize(full)
    clean_sents = postprocess_filter(raw_sents)
    print(f"  LDA filter: {len(raw_sents)} → {len(clean_sents)} sentences kept")

    return ' '.join(clean_sents)


# ── Run ───────────────────────────────────────────────────────────────
if 'model_embed' not in dir():
    from sentence_transformers import SentenceTransformer
    model_embed = SentenceTransformer('all-MiniLM-L6-v2')

for fname in os.listdir(dedup_dir):
    if not fname.endswith('.txt'):
        continue

    in_path  = os.path.join(dedup_dir, fname)
    out_path = os.path.join(summary_dir, fname)

    with open(in_path, 'r', encoding='utf-8') as f:
        sents = [ln.strip() for ln in f if ln.strip()]

    print(f"\n📄 {fname}")
    summary = summarise_full_transcript(sents, model_embed)

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(summary)

    print(f"  ✅ {len(summary.split())} words saved")
    print(f"  Preview: {summary[:140]}...")

print('\n✅ All summaries saved to summaries/')

In [ ]:
# CELL 9 (FIXED): Load question generation model (no pipeline)
# This avoids transformers.pipeline() task-name issues across versions.

from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

print("Loading question generation model...")

qg_model_name = "mrm8488/t5-base-finetuned-question-generation-ap"

qg_tokenizer = T5Tokenizer.from_pretrained(qg_model_name)
qg_model = T5ForConditionalGeneration.from_pretrained(qg_model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qg_model = qg_model.to(device)
qg_model.eval()

print(f"✓ QG model loaded on: {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# CELL 10: Context-aware quiz generation
#
# Design principles:
#   1. Questions generated from BART summary (clean prose, not raw transcript)
#   2. Answer = best noun phrase via POS tagging (2-word preferred)
#   3. Question = lmqg/t5-base-squad-qg with full summary as context
#   4. Distractors = noun phrases from OTHER summary sentences only
#      → always topic-relevant, zero hardcoded word lists
#   5. Duplicate questions filtered via embedding similarity
#   6. Question count auto-scales with summary length

import re, string, random
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sentence_transformers import SentenceTransformer, util as st_util
import torch

nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

random.seed(42)
stop_words = set(stopwords.words('english'))

# ── Load QG model ─────────────────────────────────────────────────────
# lmqg/t5-base-squad-qg: fine-tuned on SQuAD for question generation
# Input:  "generate question: <context with <hl>answer<hl> highlighted>"
# Output: a natural language question
print("Loading QG model (lmqg/t5-base-squad-qg)...")
qg_tok = T5Tokenizer.from_pretrained("lmqg/t5-base-squad-qg")
qg_mod = T5ForConditionalGeneration.from_pretrained("lmqg/t5-base-squad-qg")
qg_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qg_mod = qg_mod.to(qg_dev)
qg_mod.eval()
print(f"✅ QG model loaded on {qg_dev}")

# Reuse embedding model from earlier cell if available
if 'model_embed' not in dir():
    model_embed = SentenceTransformer('all-MiniLM-L6-v2')


def split_to_sentences(text):
    """Split summary paragraph into clean sentences (min 6 words each)."""
    return [s.strip() for s in sent_tokenize(text) if len(s.split()) >= 6]


def extract_answer(sentence):
    """
    Extract a meaningful noun phrase as the quiz answer.

    Improvements over previous version:
    1. Prefers noun phrases that are at least 2 words
       (avoids single vague words like "sorts", "number", "system")
    2. Skips noun phrases where the head noun is a pronoun or
       a quantity word (many, number, sort, kind, thing, way)
    3. Prefers noun phrases that appear in the MIDDLE or END
       of the sentence — the grammatical focus of the sentence
       rather than the subject (which is often too obvious)
    4. Caps at 4 words max (was 3, slightly more context)
    """
    import string
    import nltk
    from nltk.tokenize import word_tokenize

    # Words that make bad answers — too vague or partial
    vague_heads = {
        'number', 'sort', 'sorts', 'kind', 'kinds', 'type', 'types',
        'thing', 'things', 'way', 'ways', 'lot', 'lots', 'part',
        'parts', 'set', 'use', 'uses', 'form', 'forms', 'case',
        'cases', 'system', 'systems', 'tool', 'tools', 'means',
        'place', 'point', 'aspect', 'sense', 'level', 'amount'
    }

    try:
        tokens = word_tokenize(sentence)
        tags   = nltk.pos_tag(tokens)

        # Build all noun phrases
        phrases  = []
        current  = []
        for word, tag in tags:
            clean = word.strip(string.punctuation).lower()
            is_content = (
                (tag.startswith('NN') or tag.startswith('JJ'))
                and clean not in stop_words
                and len(clean) > 2
                and word not in string.punctuation
            )
            if is_content:
                current.append(word.strip(string.punctuation))
            else:
                if current:
                    phrases.append(' '.join(current))
                current = []
        if current:
            phrases.append(' '.join(current))

        if not phrases:
            raise ValueError("no phrases found")

        # Score each phrase
        scored = []
        for p in phrases:
            words  = p.split()
            n      = len(words)
            head   = words[-1].lower()   # head noun is last word

            if n < 1 or n > 4:
                continue
            if head in vague_heads:
                continue   # skip vague head nouns

            # Prefer 2-3 word phrases most
            length_score = {1: 4, 2: 10, 3: 9, 4: 6}.get(n, 0)

            # Prefer phrases from second half of sentence
            all_positions = [
                i for i, t in enumerate(tokens)
                if t.strip(string.punctuation) == words[0]
            ]
            pos = all_positions[0] if all_positions else 0
            position_score = 3 if pos > len(tokens) // 2 else 0

            scored.append((length_score + position_score, p))

        if scored:
            return sorted(scored, reverse=True)[0][1]

    except Exception:
        pass

    # Fallback: first non-stopword token longer than 4 chars
    for w in sentence.split():
        c = w.strip(string.punctuation).lower()
        if c and c not in stop_words and len(c) > 4 and c not in vague_heads:
            return w.strip(string.punctuation)

    return sentence.split()[0] if sentence.split() else "this"


def generate_question(sentence, answer, full_context):
    """
    Generate a question using lmqg/t5-base-squad-qg.
    Highlights the answer in the full summary context so the model
    understands what concept the question should be about.
    Strips any model output prefix and ensures proper formatting.
    """
    highlighted = full_context.replace(answer, f"<hl> {answer} <hl>", 1)
    input_text  = f"generate question: {highlighted}"

    try:
        ids = qg_tok.encode(
            input_text, return_tensors='pt', max_length=512, truncation=True
        ).to(qg_dev)

        with torch.no_grad():
            out = qg_mod.generate(
                ids, max_length=64, min_length=8,
                num_beams=4, no_repeat_ngram_size=2, early_stopping=True
            )

        q = qg_tok.decode(out[0], skip_special_tokens=True).strip()

        # Strip any prefix the model may output
        for prefix in ['question:', 'Question:', 'generate question:']:
            if q.lower().startswith(prefix.lower()):
                q = q[len(prefix):].strip()

        if not q.endswith('?'):
            q += '?'
        return q[0].upper() + q[1:] if q else q

    except Exception:
        return "Fill in the blank: " + sentence.replace(answer, '________', 1) + "?"


def get_distractors(answer, all_sentences, n=3):
    """
    Build distractors purely from noun phrases in OTHER summary sentences.
    This guarantees distractors are always about the same topic as the video.
    No hardcoded word lists of any kind.
    If the best-answer extraction doesn't yield enough candidates,
    we do a deeper pass extracting ALL noun phrases from each sentence.
    """
    answer_lower = answer.lower().strip()
    seen         = {answer_lower}
    distractors  = []

    # Pass 1: best noun phrase from each other sentence
    for sent in all_sentences:
        if answer_lower in sent.lower():
            continue
        candidate = extract_answer(sent)
        c = candidate.lower().strip()
        if c not in seen and len(candidate) > 1:
            seen.add(c)
            distractors.append(candidate)
        if len(distractors) >= n * 4:
            break

    random.shuffle(distractors)

    # Pass 2: if still not enough, extract ALL noun phrases from each sentence
    if len(distractors) < n:
        for sent in all_sentences:
            if answer_lower in sent.lower():
                continue
            try:
                tokens = word_tokenize(sent)
                tags   = nltk.pos_tag(tokens)
                chunk, current = [], []
                for word, tag in tags:
                    c = word.strip(string.punctuation).lower()
                    if tag.startswith('NN') and c not in stop_words and len(c) > 2:
                        current.append(word.strip(string.punctuation))
                    else:
                        if current:
                            chunk.append(' '.join(current))
                        current = []
                if current:
                    chunk.append(' '.join(current))
                for p in chunk:
                    p_lower = p.lower()
                    if p_lower not in seen and p_lower != answer_lower:
                        seen.add(p_lower)
                        distractors.append(p)
                    if len(distractors) >= n:
                        break
            except Exception:
                continue
            if len(distractors) >= n:
                break

    return distractors[:n] if len(distractors) >= n else distractors


def is_duplicate_question(q_new, used_questions, embed_model, threshold=0.82):
    """
    Check if a newly generated question is semantically too similar
    to any previously accepted question.
    Uses sentence embeddings — more robust than word overlap.
    """
    if not used_questions:
        return False
    new_emb  = embed_model.encode([q_new], convert_to_tensor=True)
    used_emb = embed_model.encode(list(used_questions), convert_to_tensor=True)
    sims     = st_util.cos_sim(new_emb, used_emb)
    return sims.max().item() > threshold


def generate_quiz(summary_text, num_q=None):
    """
    Generate MCQ questions from a BART summary paragraph.

    Improvements:
    1. Filters out sentences that are primarily about a named person
       (presenter introductions like "Martin Keen is a master inventor")
       rather than the subject matter — detected by checking if the
       grammatical subject is a proper noun (NNP) that appears only once
    2. Filters out sentences that are too short to yield good questions
    3. Scales question count with summary length as before
    4. Deduplication via embedding similarity as before
    """
    from nltk.tokenize import sent_tokenize
    from collections import Counter
    import nltk, string

    sentences = split_to_sentences(summary_text)
    if not sentences:
        print("  No sentences in summary.")
        return []

    # Auto-scale
    if num_q is None:
        num_q = 3 if len(sentences) < 10 else (5 if len(sentences) <= 20 else 7)

    print(f"  Summary: {len(sentences)} sentences → {num_q} questions")

    # ── Filter: remove presenter-introduction sentences ───────────────
    all_proper = []
    sent_subjects = []
    for sent in sentences:
        try:
            tokens = nltk.word_tokenize(sent)
            tags   = nltk.pos_tag(tokens)
            # Subject = first NNP/NNPS sequence at start of sentence
            subject_nouns = []
            for word, tag in tags[:6]:
                if tag in ('NNP', 'NNPS'):
                    subject_nouns.append(word.lower())
                elif subject_nouns:
                    break
            sent_subjects.append(subject_nouns)
            all_proper.extend(subject_nouns)
        except Exception:
            sent_subjects.append([])

    proper_freq   = Counter(all_proper)
    unique_proper = {n for n, c in proper_freq.items() if c == 1}

    def is_presenter_sentence(idx):
        subj = sent_subjects[idx]
        if not subj:
            return False
        return all(n in unique_proper for n in subj) and len(subj) >= 1

    filtered = []
    for i, sent in enumerate(sentences):
        if is_presenter_sentence(i):
            print(f"  Skipping presenter sentence: {sent[:55]}...")
            continue
        if len(sent.split()) < 8:
            continue
        filtered.append(sent)

    if not filtered:
        filtered = sentences

    # Deduplicate by embedding similarity
    embs         = model_embed.encode(filtered, convert_to_tensor=True)
    unique_sents = []
    unique_embs  = []
    for i, sent in enumerate(filtered):
        if not unique_embs:
            unique_sents.append(sent)
            unique_embs.append(embs[i])
        else:
            sim = st_util.cos_sim(
                embs[i].unsqueeze(0), torch.stack(unique_embs)
            ).max().item()
            if sim < 0.75:
                unique_sents.append(sent)
                unique_embs.append(embs[i])

    print(f"  Candidates after filtering: {len(unique_sents)}")
    candidates = unique_sents[:num_q + 2]
    used_qs    = set()
    questions  = []

    for sent in candidates:
        if len(questions) >= num_q:
            break
        print(f"  Q{len(questions)+1}: {sent[:55]}...")

        answer   = extract_answer(sent)
        question = generate_question(sent, answer, summary_text)

        if is_duplicate_question(question, used_qs, model_embed, threshold=0.82):
            print("    Duplicate, skipping")
            continue

        used_qs.add(question)
        distractors = get_distractors(answer, unique_sents, n=3)
        options     = [answer] + distractors
        random.shuffle(options)

        questions.append({
            'question':        question,
            'answer':          answer,
            'options':         options,
            'type':            'MCQ',
            'source_sentence': sent
        })

    print(f"  ✅ {len(questions)} questions generated")
    return questions


print('✅ Quiz generation ready')
print('   Answers:     POS-based noun phrase extraction')
print('   Questions:   lmqg/t5-base-squad-qg with full context')
print('   Distractors: content-derived from summary sentences only')
print('   Dedup:       embedding similarity (no word-overlap heuristics)')

In [ ]:
# CELL 11: Generate quizzes for all summaries (writes quiz_output/*.json)

quiz_output_dir = 'quiz_output'
os.makedirs(quiz_output_dir, exist_ok=True)

summary_files = [f for f in os.listdir(summary_dir) if f.endswith('.txt')]
print('Summary files found:', len(summary_files))

for fname in summary_files:
    in_path = os.path.join(summary_dir, fname)

    with open(in_path, 'r', encoding='utf-8') as f:
        summary_text = f.read().strip()

    print('\nGenerating quiz for:', fname)
    # Pass full summary TEXT (paragraph) to generate_quiz
    # generate_quiz handles splitting into sentences internally
    questions = generate_quiz(summary_text, num_q=None)  # auto-scales with summary length

    out = {
        'video': fname,
        'summary': summary_text,   # now a paragraph string, not a list
        'questions': questions
    }

    out_path = os.path.join(quiz_output_dir, fname.replace('.txt', '.json'))
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2)

    print('✅ Saved:', out_path, '| Qs:', len(questions))

print('\n✅ All quizzes generated')

In [ ]:
# CELL 12: Evaluate summaries with ROUGE (summary vs transcript)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_results = {}

print(f"{'File':<35} | {'ROUGE-1':>8} | {'ROUGE-2':>8} | {'ROUGE-L':>8}")
print('-' * 70)

for fname in [f for f in os.listdir(summary_dir) if f.endswith('.txt')]:
    t_path = os.path.join(transcript_dir, fname)
    s_path = os.path.join(summary_dir, fname)
    if not os.path.exists(t_path):
        continue

    with open(t_path, 'r', encoding='utf-8') as f:
        transcript_text = f.read()
    with open(s_path, 'r', encoding='utf-8') as f:
        summary_text = ' '.join([ln.strip() for ln in f if ln.strip()])

    scores = scorer.score(transcript_text, summary_text)
    rouge_results[fname] = {
        'rouge1_f1': round(scores['rouge1'].fmeasure, 4),
        'rouge2_f1': round(scores['rouge2'].fmeasure, 4),
        'rougeL_f1': round(scores['rougeL'].fmeasure, 4)
    }

    print(f"{fname:<35} | {scores['rouge1'].fmeasure:>8.4f} | {scores['rouge2'].fmeasure:>8.4f} | {scores['rougeL'].fmeasure:>8.4f}")

if rouge_results:
    avg_r1 = sum(v['rouge1_f1'] for v in rouge_results.values()) / len(rouge_results)
    avg_r2 = sum(v['rouge2_f1'] for v in rouge_results.values()) / len(rouge_results)
    avg_rl = sum(v['rougeL_f1'] for v in rouge_results.values()) / len(rouge_results)
    print('-' * 70)
    print(f"{'AVERAGE':<35} | {avg_r1:>8.4f} | {avg_r2:>8.4f} | {avg_rl:>8.4f}")

In [ ]:
# CELL 13: Save evaluation report to evaluation/evaluation_report.json
# (Includes ROUGE + compression ratio + redundancy reduction)

evaluation_dir = 'evaluation'
os.makedirs(evaluation_dir, exist_ok=True)

report = []
for fname, rr in rouge_results.items():
    summary_path = os.path.join(summary_dir, fname)
    transcript_path = os.path.join(transcript_dir, fname)
    seg_path = os.path.join(segmented_dir, fname)
    dedup_path = os.path.join(dedup_dir, fname)

    with open(summary_path, 'r', encoding='utf-8') as f:
        summary_words = len(f.read().split())
    with open(transcript_path, 'r', encoding='utf-8') as f:
        transcript_words = len(f.read().split())

    seg_count = sum(1 for _ in open(seg_path, 'r', encoding='utf-8')) if os.path.exists(seg_path) else 0
    dedup_count = sum(1 for _ in open(dedup_path, 'r', encoding='utf-8')) if os.path.exists(dedup_path) else 0

    compression = round((summary_words / transcript_words) * 100, 2) if transcript_words else 0
    redundancy_removed = round(((seg_count - dedup_count) / seg_count) * 100, 2) if seg_count else 0

    report.append({
        'file': fname,
        **rr,
        'summary_words': summary_words,
        'transcript_words': transcript_words,
        'compression_ratio_pct': compression,
        'redundancy_reduction_pct': redundancy_removed
    })

out_path = os.path.join(evaluation_dir, 'evaluation_report.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print('✅ Saved:', out_path)
print('Entries:', len(report))

In [ ]:
# CELL 14: ★ FINAL DEMO DISPLAY (faculty-friendly)

quiz_output_dir = 'quiz_output'
evaluation_dir = 'evaluation'

# Load evaluation report
eval_report = {}
eval_path = os.path.join(evaluation_dir, 'evaluation_report.json')
if os.path.exists(eval_path):
    with open(eval_path, 'r', encoding='utf-8') as f:
        for entry in json.load(f):
            eval_report[entry['file']] = entry

quiz_files = [f for f in os.listdir(quiz_output_dir) if f.endswith('.json')]
if not quiz_files:
    print('No quiz files found. Run quiz generation first.')
else:
    for qfile in quiz_files:
        txt_name = qfile.replace('.json', '.txt')
        with open(os.path.join(quiz_output_dir, qfile), 'r', encoding='utf-8') as f:
            data = json.load(f)

        print('\n' + '=' * 70)
        print('VIDEO:', data['video'])
        print('=' * 70)

        # Transcript preview
        t_path = os.path.join(transcript_dir, txt_name)
        if os.path.exists(t_path):
            with open(t_path, 'r', encoding='utf-8') as f:
                raw = f.read().split('. ')
            print('\nTRANSCRIPT PREVIEW (first 3 sentences):')
            for i, s in enumerate(raw[:3], 1):
                print(f'  {i}. {s.strip()}')

        # Summary
        summary_text = data['summary']
        if isinstance(summary_text, list):
            summary_text = ' '.join(summary_text)
        # Split into sentences for display
        from nltk.tokenize import sent_tokenize
        summary_sents = sent_tokenize(summary_text)
        print(f"\nSUMMARY ({len(summary_sents)} sentences, {len(summary_text.split())} words):")
        for i, s in enumerate(summary_sents, 1):
            print(f'  {i}. {s}')

        # Quiz
        print('\nQUIZ QUESTIONS:')
        for i, q in enumerate(data['questions'], 1):
            print(f"\n  Q{i}: {q['question']}")
            labels = ['A', 'B', 'C', 'D']
            for label, opt in zip(labels, q.get('options', [])):
                mark = ' (correct)' if opt == q['answer'] else ''
                print(f'     {label}) {opt}{mark}')

        # Evaluation
        if txt_name in eval_report:
            ev = eval_report[txt_name]
            print('\nEVALUATION:')
            print('  ROUGE-1 F1:', ev['rouge1_f1'])
            print('  ROUGE-2 F1:', ev['rouge2_f1'])
            print('  ROUGE-L F1:', ev['rougeL_f1'])
            print('  Compression %:', ev['compression_ratio_pct'])
            print('  Redundancy removed %:', ev['redundancy_reduction_pct'])

        print('\n' + '=' * 70)

    if eval_report:
        avg_r1 = sum(v['rouge1_f1'] for v in eval_report.values()) / len(eval_report)
        print(f"\n✅ Pipeline complete! Processed {len(quiz_files)} video(s).")
        print('Average ROUGE-1 F1:', round(avg_r1, 4))

In [ ]:
# RESET CELL (no URL): clear outputs only

import os, shutil

for d in ["youtube_videos", "audio_files", "transcripts", "segmented", "deduplicated", "summaries", "quiz_output", "evaluation"]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

print("✅ Cleared old outputs. Now change YOUTUBE_URL in the download cell and run it, then run from CELL 4 onward.")

## 🎓 Demo Instructions (tomorrow)

**Recommended demo (fast & safe):**
1. Set **GPU runtime** (T4).
2. Set `SOURCE = 'youtube'` in **Cell 2**.
3. Change `YOUTUBE_URL` in **Cell 3A** to a short lecture.
4. Run **Cell 1 → Cell 14** in order once (tonight) so outputs are saved.
5. During faculty demo, run **Cell 14 only** to instantly show results.

**If YouTube download is blocked:** set `SOURCE='kaggle'` and run Cell 3B instead.
